In [2]:
# Domain Adaptation Model Testing Notebook
# Load and test trained DA models with comprehensive evaluation

import sys
sys.path.append("..")
import os

import torch
import torch.nn.functional as F
import numpy as np
import yaml
import pandas as pd
from pathlib import Path

from src.models.base_models import create_model
from src.models.domain_discriminator import DomainDiscriminator, create_da_model
from src.data.numu_dataset import NuMuDataset, create_numu_dataloader_from_ds

In [3]:
PATH2MODELS = Path("../SharedModels")
TASK_NAME = "AllNuMu"
MODEL_NAME = "da_numu_251108_middle_constlambda_aug"
EXPERIMENT_DIR = Path(f"../experiments/numu/{MODEL_NAME}")

path_to_share_model = PATH2MODELS / TASK_NAME / MODEL_NAME
os.makedirs(path_to_share_model, exist_ok=True)

In [4]:
# Load configuration
config_path = EXPERIMENT_DIR / "da_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Load trained models from best checkpoint
checkpoint_path = EXPERIMENT_DIR / "best_da_model.pth"
print(f"Loading checkpoint from: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path)

# Create and load base model
model = create_model(config['model'])
model.load_state_dict(checkpoint['base_model_state_dict'])
model.eval()


Loading checkpoint from: ../experiments/numu/da_numu_251108_middle_constlambda_aug/best_da_model.pth


/tmp/ipykernel_2205922/1285245099.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


NuMuClassifierModel(
  (feature_extractor): AttentionFeatureExtractor(
    (input_projection): Linear(in_features=5, out_features=128, bias=True)
    (pos_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer_encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (layer_n

In [6]:
# Save base model (classifier) to shared directory
print(f"Saving base model to: {path_to_share_model}")

# Save model state dict only
model_save_path = path_to_share_model / "base_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to: {model_save_path}")

# Save model configuration (minimal config for loading)
model_config = {
    'model': config['model'],
    'experiment_name': MODEL_NAME,
    'model_type': 'base_classifier',
    'description': 'Base classifier from domain adaptation training (MC→Exp transfer)',
    'architecture': 'Transformer encoder + Binary classifier',
    'parameters': 541633,
    'input_features': 5,  # [amplitude, time, x, y, z]
    'output': 'binary_logits',
    'task': 'neutrino_detection'
}

config_save_path = path_to_share_model / "model_config.yaml"
with open(config_save_path, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)
print(f"Config saved to: {config_save_path}")

# Copy normalization config for consistent preprocessing
norm_config_path = EXPERIMENT_DIR / "normalization_config.yaml"
if norm_config_path.exists():
    import shutil
    shutil.copy(norm_config_path, path_to_share_model / "normalization_config.yaml")
    print(f"Normalization config copied to shared directory")

print(f"\n✅ Base model successfully exported to: {path_to_share_model}")
print(f"📁 Contents:")
for file in path_to_share_model.iterdir():
    print(f"  - {file.name}")

Saving base model to: ../SharedModels/AllNuMu/da_numu_251108_middle_constlambda_aug
Model saved to: ../SharedModels/AllNuMu/da_numu_251108_middle_constlambda_aug/base_model.pth
Config saved to: ../SharedModels/AllNuMu/da_numu_251108_middle_constlambda_aug/model_config.yaml
Normalization config copied to shared directory

✅ Base model successfully exported to: ../SharedModels/AllNuMu/da_numu_251108_middle_constlambda_aug
📁 Contents:
  - README.md
  - base_model.pth
  - model_config.yaml
  - normalization_config.yaml


In [13]:
from SharedModels.AllNuMu.da_numu_251108_middle_constlambda_aug.src.base_models import create_model

# Test loading the saved model to verify it works
print("\n🧪 Testing model loading...")

# Load config
with open(path_to_share_model / "model_config.yaml", 'r') as f:
    test_config = yaml.safe_load(f)

# Create new model instance
test_model = create_model(test_config['model'])

# Load weights
test_model.load_state_dict(torch.load(path_to_share_model / "base_model.pth"))
test_model.eval()

print("✅ Model loaded successfully!")
print(f"📊 Parameters: {sum(p.numel() for p in test_model.parameters()):,}")

# Test with dummy input
dummy_input = {
    'features': torch.randn(10, 100, 5),  # [batch_size, seq_len, features]
    'lengths': torch.tensor([100]*10),       # actual sequence length
    'mask': torch.ones(10,100, dtype=bool)
}

with torch.no_grad():
    output = test_model(dummy_input)
    print(f"🔍 Test output shape: {output.shape}")
    print(f"📈 Test output range: [{output.min().item():.3f}, {output.max().item():.3f}]")
    
print("\n✅ Model export and testing completed successfully!")


🧪 Testing model loading...
✅ Model loaded successfully!
📊 Parameters: 541,633
🔍 Test output shape: torch.Size([10, 1])
📈 Test output range: [13.437, 16.166]

✅ Model export and testing completed successfully!


/tmp/ipykernel_2205922/1550997563.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_model.load_state_dict(torch.load(path_to_share_model / "base_model.pth"))


In [14]:
output

tensor([[15.3057],
        [15.8859],
        [14.5254],
        [16.1661],
        [14.9699],
        [15.7227],
        [15.1875],
        [13.4369],
        [14.5831],
        [15.5362]])